# Automation and workflowing

A rule says what to do with which messages. `automate` looks through the inbox and
acts once per message; run it from cron, a timer, or an Ansible play.

In [1]:
from amoorion import Amoorion

from amoorion.models import Message

# A sandbox, with its own inbox.
amo = Amoorion()

# What hooks would have delivered.
amo.store.keep(
    [
        Message(
            platform="hook:generic",
            text="deploy finished",
            id="1",
        ),
        Message(
            platform="hook:generic",
            text="backup finished",
            id="2",
        ),
    ]
)

# A rule: which messages, and what steps. `$text` is the text of the message.
rule = {
    "name": "announce-deploys",
    "platform": "hook:generic",
    "contains": "deploy",
    "steps": [
        {
            "method": "publish",
            "params": {
                "text": "Status: $text",
                "to": ["dryrun"],
            },
        }
    ],
}

# It runs once per matching message.
for done in amo.automate(rules=[rule]):
    print(
        done["rule"],
        done["message_id"],
        done["ok"],
        done["steps"][0]["result"][0]["message"]["text"],
    )

announce-deploys 1 True Status: deploy finished


Run again, it finds nothing new:

In [ ]:
# The second run finds nothing new: this is safe to run from a timer.
# amo.automate(rules=[rule])

[]

A step that fails leaves the message unhandled, so the next run tries again:

In [ ]:
# A new message…
amo.store.keep(
    [
        Message(
            platform="hook:generic",
            text="deploy two",
            id="3",
        )
    ]
)

# …and a rule that can't succeed: this platform doesn't exist.
broken = rule | {
    "name": "to-nowhere",
    "contains": "two",
    "steps": [
        {
            "method": "publish",
            "params": {"text": "$text", "to": ["nowhere"]},
        }  # this can still be used with workflows i think
    ],
}

# It fails…
print([done["ok"] for done in amo.automate(rules=[broken])])

# …and is tried again, as failures aren't marked as handled.
print([done["ok"] for done in amo.automate(rules=[broken])])

[False]
[False]


Rules are plain data, so they can live in YAML next to `amo.save()`:

In [ ]:
import yaml

# A rule is plain data: this is its form in a file.

# alright, so let's assum it that all rules are yaml files defined and readable
# output of workflow should have logs then
# and result can be artifact of json for example that can be passed to the sender of post on socialmedia

print(
    yaml.safe_dump(
        {"rules": [rule]},
        sort_keys=False,
    )
)

rules:
- name: announce-deploys
  platform: hook:generic
  contains: deploy
  steps:
  - method: publish
    params:
      text: 'Status: $text'
      to:
      - dryrun

